# State of Data Brasil — Camada Silver

In [1]:
%idle_timeout 2880
%glue_version 5.1
%worker_type G.1X
%number_of_workers 5

from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql import functions as F

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 5.1
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: 4b00db95-53a7-46e4-b426-1dc96a4544a9
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 4b00db95-53a7-46e4-b426-1dc96a4544a9 to get into ready status...
Session 4b00db95-53a7-46e4-b426-1dc96a4544a9 

## Configuração

In [2]:
DATABASE_NAME = "state_of_data"
BUCKET_NAME = "data-6625-3564-2976"

BRONZE_TABLES = {
    "2023": "tb_state_data_2023_bronze",
    "2024": "tb_state_data_2024_bronze",
    "2025_2026": "tb_state_data_2025_2026_bronze"
}

SILVER_TABLE = f"{DATABASE_NAME}.tb_state_data_silver"
SILVER_PATH = f"s3://{BUCKET_NAME}/data-output/silver/state-of-data/"

## Leitura da Bronze

In [3]:
df_2023 = glueContext.create_dynamic_frame.from_catalog(
    database=DATABASE_NAME,
    table_name=BRONZE_TABLES["2023"]
).toDF()

df_2024 = glueContext.create_dynamic_frame.from_catalog(
    database=DATABASE_NAME,
    table_name=BRONZE_TABLES["2024"]
).toDF()

df_2025 = glueContext.create_dynamic_frame.from_catalog(
    database=DATABASE_NAME,
    table_name=BRONZE_TABLES["2025_2026"]
).toDF()

/usr/lib/spark/python/lib/pyspark.zip/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.


## Limpando e analisando a bronze

In [4]:
fontes_bronze = [
    ("2023", df_2023, "p0_id"),
    ("2024", df_2024, "col_0_a_token"),
    ("2025_2026", df_2025, "col_0_a_token")
]

for ano, df, coluna_id in fontes_bronze:
    resumo = (
        df
        .agg(
            F.count("*").alias("registros"),
            F.countDistinct(coluna_id).alias("ids_unicos"),
            F.sum(
                F.when(
                    F.col(coluna_id).isNull()
                    | (F.trim(F.col(coluna_id)) == ""),
                    1
                ).otherwise(0)
            ).alias("ids_vazios")
        )
        .first()
    )

    duplicados = resumo["registros"] - resumo["ids_unicos"] - resumo["ids_vazios"]

    print(
        ano,
        "| registros:", resumo["registros"],
        "| ids únicos:", resumo["ids_unicos"],
        "| ids vazios:", resumo["ids_vazios"],
        "| duplicados:", duplicados
    )

2023 | registros: 5293 | ids únicos: 5293 | ids vazios: 0 | duplicados: 0
2024 | registros: 5217 | ids únicos: 5215 | ids vazios: 0 | duplicados: 2
2025_2026 | registros: 3495 | ids únicos: 3494 | ids vazios: 0 | duplicados: 1


In [5]:
campos_auditoria = {
    "2023": {
        "genero": "p1_b_genero",
        "senioridade": "p2_g_nivel",
        "faixa_salarial": "p2_h_faixa_salarial",
        "cloud_preferida": "p4_i_cloud_preferida",
        "bancos_dados": "p4_g_quais_dos_bancos_de_dados_fontes_de_dados_listados_abaixo_voce_utiliza_no_trabalho",
        "ferramenta_bi": "p4_j_ferramenta_de_bi_utilizada_no_dia_a_dia"
    },
    "2024": {
        "genero": "col_1_b_genero",
        "senioridade": "col_2_g_nivel",
        "faixa_salarial": "col_2_h_faixa_salarial",
        "cloud_preferida": "col_4_i_cloud_preferida",
        "bancos_dados": "col_4_g_banco_de_dados_dia_a_dia",
        "ferramenta_bi": "col_4_j_ferramenta_de_bi_dia_a_dia"
    },
    "2025_2026": {
        "genero": "col_1_b_genero",
        "senioridade": "col_2_g_nivel",
        "faixa_salarial": "col_2_h_faixa_salarial",
        "cloud_preferida": "col_4_f_cloud_preferida",
        "bancos_dados": "col_4_d_banco_de_dados_dia_a_dia",
        "ferramenta_bi": "col_4_g_ferramenta_de_bi_dia_a_dia"
    }
}

dfs_auditoria = {
    "2023": df_2023,
    "2024": df_2024,
    "2025_2026": df_2025
}

for ano, campos in campos_auditoria.items():
    df = dfs_auditoria[ano]

    expressoes = []

    for nome, coluna in campos.items():
        expressoes.extend([
            F.sum(
                F.when(
                    F.col(coluna).isNull() | (F.trim(F.col(coluna)) == ""),
                    1
                ).otherwise(0)
            ).alias(f"{nome}_vazios"),
            F.sum(
                F.when(
                    F.col(coluna).isNotNull() & (F.trim(F.col(coluna)) != ""),
                    1
                ).otherwise(0)
            ).alias(f"{nome}_preenchidos")
        ])

    resultado = df.agg(*expressoes).first().asDict()

    print(ano)

    for nome in campos:
        print(
            nome,
            "| preenchidos:", resultado[f"{nome}_preenchidos"],
            "| vazios:", resultado[f"{nome}_vazios"]
        )

    print()

2023
genero | preenchidos: 5293 | vazios: 0
senioridade | preenchidos: 3857 | vazios: 1436
faixa_salarial | preenchidos: 4753 | vazios: 540
cloud_preferida | preenchidos: 3772 | vazios: 1521
bancos_dados | preenchidos: 3772 | vazios: 1521
ferramenta_bi | preenchidos: 3772 | vazios: 1521

2024
genero | preenchidos: 5217 | vazios: 0
senioridade | preenchidos: 3818 | vazios: 1399
faixa_salarial | preenchidos: 4863 | vazios: 354
cloud_preferida | preenchidos: 3589 | vazios: 1628
bancos_dados | preenchidos: 3589 | vazios: 1628
ferramenta_bi | preenchidos: 3619 | vazios: 1598

2025_2026
genero | preenchidos: 3495 | vazios: 0
senioridade | preenchidos: 2501 | vazios: 994
faixa_salarial | preenchidos: 3228 | vazios: 267
cloud_preferida | preenchidos: 2096 | vazios: 1399
bancos_dados | preenchidos: 2096 | vazios: 1399
ferramenta_bi | preenchidos: 2106 | vazios: 1389


In [6]:
campos_multiplos = {
    "2023": {
        "bancos_dados": "p4_g_quais_dos_bancos_de_dados_fontes_de_dados_listados_abaixo_voce_utiliza_no_trabalho",
        "ferramenta_bi": "p4_j_ferramenta_de_bi_utilizada_no_dia_a_dia",
        "uso_ia_trabalho": "p4_m_utiliza_chatgpt_ou_llms_no_trabalho"
    },
    "2024": {
        "bancos_dados": "col_4_g_banco_de_dados_dia_a_dia",
        "ferramenta_bi": "col_4_j_ferramenta_de_bi_dia_a_dia",
        "uso_ia_trabalho": "col_4_m_usa_chatgpt_ou_copilot_no_trabalho"
    },
    "2025_2026": {
        "linguagem_preferida": "col_4_c_linguagem_preferida",
        "bancos_dados": "col_4_d_banco_de_dados_dia_a_dia",
        "ferramenta_bi": "col_4_g_ferramenta_de_bi_dia_a_dia",
        "uso_ia_trabalho": "col_4_j_usa_chatgpt_ou_copilot_no_trabalho"
    }
}

for ano, campos in campos_multiplos.items():
    df = dfs_auditoria[ano]

    expressoes = []

    for nome, coluna in campos.items():
        expressoes.extend([
            F.sum(
                F.when(F.col(coluna).contains(","), 1).otherwise(0)
            ).alias(f"{nome}_virgula"),
            F.sum(
                F.when(F.col(coluna).contains(";"), 1).otherwise(0)
            ).alias(f"{nome}_ponto_virgula")
        ])

    resultado = df.agg(*expressoes).first().asDict()

    print(ano)

    for nome in campos:
        print(
            nome,
            "| com vírgula:", resultado[f"{nome}_virgula"],
            "| com ;:", resultado[f"{nome}_ponto_virgula"]
        )

    print()

2023
bancos_dados | com vírgula: 2376 | com ;: 1
ferramenta_bi | com vírgula: 1811 | com ;: 0
uso_ia_trabalho | com vírgula: 3054 | com ;: 0

2024
bancos_dados | com vírgula: 2303 | com ;: 0
ferramenta_bi | com vírgula: 1695 | com ;: 0
uso_ia_trabalho | com vírgula: 3395 | com ;: 0

2025_2026
linguagem_preferida | com vírgula: 1710 | com ;: 0
bancos_dados | com vírgula: 1512 | com ;: 0
ferramenta_bi | com vírgula: 956 | com ;: 0
uso_ia_trabalho | com vírgula: 2067 | com ;: 0


Mapeando os campos

In [7]:
mapa_2023 = {
    "id_respondente": "p0_id",
    "idade": "p1_a_idade",
    "funcao_atuacao": "p4_a_1_atuacao",
    "fontes_dados": "p4_b_quais_das_fontes_de_dados_listadas_voce_ja_analisou_ou_processou_no_trabalho",
    "faixa_idade": "p1_a_1_faixa_idade",
    "genero": "p1_b_genero",
    "raca_etnia": "p1_c_cor_raca_etnia",
    "pcd": "p1_d_pcd",
    "estado": "p1_i_estado_onde_mora",
    "uf": "p1_i_1_uf_onde_mora",
    "regiao": "p1_i_2_regiao_onde_mora",
    "nivel_ensino": "p1_l_nivel_de_ensino",
    "area_formacao": "p1_m_area_de_formacao",
    "situacao_trabalho": "p2_a_qual_sua_situacao_atual_de_trabalho",
    "setor": "p2_b_setor",
    "numero_funcionarios": "p2_c_numero_de_funcionarios",
    "atua_como_gestor": "p2_d_gestor",
    "cargo_gestor": "p2_e_cargo_como_gestor",
    "cargo_atual": "p2_f_cargo_atual",
    "senioridade": "p2_g_nivel",
    "faixa_salarial": "p2_h_faixa_salarial",
    "experiencia_dados": "p2_i_quanto_tempo_de_experiencia_na_area_de_dados_voce_tem",
    "experiencia_ti": "p2_j_quanto_tempo_de_experiencia_na_area_de_ti_engenharia_de_software_voce_teve_antes_de_comecar_a_trabalhar_na_area_de_dados",
    "satisfeito": "p2_k_voce_esta_satisfeito_na_sua_empresa_atual",
    "modelo_trabalho_atual": "p2_r_atualmente_qual_a_sua_forma_de_trabalho",
    "modelo_trabalho_ideal": "p2_s_qual_a_forma_de_trabalho_ideal_para_voce",
    "linguagem_preferida": "p4_f_entre_as_linguagens_listadas_abaixo_qual_e_a_sua_preferida",
    "bancos_dados": "p4_g_quais_dos_bancos_de_dados_fontes_de_dados_listados_abaixo_voce_utiliza_no_trabalho",
    "cloud_preferida": "p4_i_cloud_preferida",
    "ferramenta_bi": "p4_j_ferramenta_de_bi_utilizada_no_dia_a_dia",
    "bi_preferida": "p4_k_qual_sua_ferramenta_de_bi_preferida",
    "ia_prioridade_empresa": "p3_e_ai_generativa_e_uma_prioridade_em_sua_empresa",
    "uso_ia_empresa": "p3_f_tipos_de_uso_de_ai_generativa_e_llms_na_empresa",
    "uso_ia_trabalho": "p4_m_utiliza_chatgpt_ou_llms_no_trabalho"
}

mapa_2024 = {
    "id_respondente": "col_0_a_token",
    "idade": "col_1_a_idade",
    "funcao_atuacao": "col_4_a_1_atuacao_em_dados",
    "fontes_dados": "col_4_b_fontes_de_dados_dia_a_dia",
    "faixa_idade": "col_1_a_1_faixa_idade",
    "genero": "col_1_b_genero",
    "raca_etnia": "col_1_c_cor_raca_etnia",
    "pcd": "col_1_d_pcd",
    "estado": "col_1_i_estado_onde_mora",
    "uf": "col_1_i_1_uf_onde_mora",
    "regiao": "col_1_i_2_regiao_onde_mora",
    "nivel_ensino": "col_1_l_nivel_de_ensino",
    "area_formacao": "col_1_m_area_de_formacao",
    "situacao_trabalho": "col_2_a_situacao_de_trabalho",
    "setor": "col_2_b_setor",
    "numero_funcionarios": "col_2_c_numero_de_funcionarios",
    "atua_como_gestor": "col_2_d_atua_como_gestor",
    "cargo_gestor": "col_2_e_cargo_como_gestor",
    "cargo_atual": "col_2_f_cargo_atual",
    "senioridade": "col_2_g_nivel",
    "faixa_salarial": "col_2_h_faixa_salarial",
    "experiencia_dados": "col_2_i_tempo_de_experiencia_em_dados",
    "experiencia_ti": "col_2_j_tempo_de_experiencia_em_ti",
    "satisfeito": "col_2_k_satisfeito_atualmente",
    "modelo_trabalho_atual": "col_2_r_modelo_de_trabalho_atual",
    "modelo_trabalho_ideal": "col_2_s_modelo_de_trabalho_ideal",
    "linguagem_preferida": "col_4_f_linguagem_preferida",
    "bancos_dados": "col_4_g_banco_de_dados_dia_a_dia",
    "cloud_preferida": "col_4_i_cloud_preferida",
    "ferramenta_bi": "col_4_j_ferramenta_de_bi_dia_a_dia",
    "bi_preferida": "col_4_k_ferramenta_de_bi_preferida",
    "ia_prioridade_empresa": "col_3_e_ai_generativa_e_llm_e_uma_prioridade",
    "uso_ia_empresa": "col_3_f_tipo_de_uso_de_ai_generativa_e_llm_na_empresa",
    "uso_ia_trabalho": "col_4_m_usa_chatgpt_ou_copilot_no_trabalho"
}

mapa_2025 = {
    "id_respondente": "col_0_a_token",
    "idade": "col_1_a_idade",
    "funcao_atuacao": "col_4_a_1_atuacao_em_dados",
    "fontes_dados": "col_4_b_fontes_de_dados_dia_a_dia",
    "faixa_idade": "col_1_a_1_faixa_idade",
    "genero": "col_1_b_genero",
    "raca_etnia": "col_1_c_cor_raca_etnia",
    "pcd": "col_1_d_pcd",
    "estado": "col_1_i_estado_onde_mora",
    "uf": "col_1_i_1_uf_onde_mora",
    "regiao": "col_1_i_2_regiao_onde_mora",
    "nivel_ensino": "col_1_l_nivel_de_ensino",
    "area_formacao": "col_1_m_area_de_formacao",
    "situacao_trabalho": "col_2_a_situacao_de_trabalho",
    "setor": "col_2_b_setor",
    "numero_funcionarios": "col_2_c_numero_de_funcionarios",
    "atua_como_gestor": "col_2_d_atua_como_gestor",
    "cargo_gestor": "col_2_e_cargo_como_gestor",
    "cargo_atual": "col_2_f_cargo_atual",
    "senioridade": "col_2_g_nivel",
    "faixa_salarial": "col_2_h_faixa_salarial",
    "experiencia_dados": "col_2_i_tempo_de_experiencia_em_dados",
    "experiencia_ti": "col_2_j_tempo_de_experiencia_em_ti",
    "satisfeito": "col_2_k_satisfeito_atualmente",
    "modelo_trabalho_atual": "col_2_q_modelo_de_trabalho_atual",
    "modelo_trabalho_ideal": "col_2_r_modelo_de_trabalho_ideal",
    "linguagem_preferida": "col_4_c_linguagem_preferida",
    "bancos_dados": "col_4_d_banco_de_dados_dia_a_dia",
    "cloud_preferida": "col_4_f_cloud_preferida",
    "ferramenta_bi": "col_4_g_ferramenta_de_bi_dia_a_dia",
    "bi_preferida": "col_4_h_ferramenta_de_bi_preferida",
    "ia_prioridade_empresa": "col_3_e_ai_generativa_e_llm_e_uma_prioridade",
    "uso_ia_empresa": "col_3_f_tipo_de_uso_de_ai_generativa_e_llm_na_empresa",
    "uso_ia_trabalho": "col_4_j_usa_chatgpt_ou_copilot_no_trabalho"
}

configuracoes = [
    ("2023", df_2023, mapa_2023),
    ("2024", df_2024, mapa_2024),
    ("2025_2026", df_2025, mapa_2025)
]

for ano, df, mapa in configuracoes:
    faltantes = [coluna for coluna in mapa.values() if coluna not in df.columns]
    if faltantes:
        raise ValueError(f"{ano}: colunas não encontradas: {faltantes}")

## Padronização

In [8]:
def normalizar_booleano(coluna):
    valor = F.lower(F.trim(F.col(coluna).cast("string")))

    return (
        F.when(valor.isin("1", "1.0", "true", "sim"), F.lit(True))
        .when(valor.isin("0", "0.0", "false", "não", "nao"), F.lit(False))
        .otherwise(F.lit(None).cast("boolean"))
    )

def normalizar_tecnologia(coluna):
    valor = F.trim(coluna)
    chave = F.lower(valor)

    return (
        F.when(chave == "sql", "SQL")
        .when(chave == "sql server", "SQL Server")
        .when(chave.isin("go", "golang"), "Go")
        .when(chave == "pyspark", "PySpark")
        .when(chave == "dax", "DAX")
        .when(chave.isin("superset", "apache superset"), "Superset")
        .when(chave == "thoughtspot", "ThoughtSpot")
        .when(chave == "microsoft powerbi", "Microsoft Power BI")
        .when(chave == "amazon quicksight", "Amazon QuickSight")
        .when(chave == "javascript", "JavaScript")
        .otherwise(valor)
    )

def normalizar_lista(coluna):
    itens = F.split(
        F.regexp_replace(F.col(coluna), r"\s*;\s*", ", "),
        r"\s*,\s*"
    )

    itens = F.transform(
        itens,
        lambda x: normalizar_tecnologia(x)
    )

    return F.concat_ws(
        ", ",
        F.array_distinct(
            F.filter(
                itens,
                lambda x: x.isNotNull() & (F.length(F.trim(x)) > 0)
            )
        )
    )

def criar_silver(df, mapa, ano):
    df = df.dropDuplicates()

    df = df.select(
        *[
            F.col(origem).cast("string").alias(destino)
            for destino, origem in mapa.items()
        ]
    )

    df = df.withColumn("ano_pesquisa", F.lit(ano))

    for coluna, tipo in df.dtypes:
        if tipo == "string":
            df = df.withColumn(
                coluna,
                F.when(
                    F.trim(F.col(coluna)) == "",
                    F.lit(None)
                ).otherwise(F.trim(F.col(coluna)))
            )

    return (
        df
        .withColumn("idade", F.col("idade").cast("int"))
        .withColumn("atua_como_gestor", normalizar_booleano("atua_como_gestor"))
        .withColumn("satisfeito", normalizar_booleano("satisfeito"))
        .withColumn("faixa_salarial_raw", F.col("faixa_salarial"))
        .withColumn("linguagem_preferida_raw", F.col("linguagem_preferida"))
        .withColumn("bancos_dados_raw", F.col("bancos_dados"))
        .withColumn("cloud_preferida_raw", F.col("cloud_preferida"))
        .withColumn("ferramenta_bi_raw", F.col("ferramenta_bi"))
        .withColumn("bi_preferida_raw", F.col("bi_preferida"))
        .withColumn(
            "faixa_salarial",
            F.when(
                F.col("faixa_salarial") == "de R$ 101/mês a R$ 2.000/mês",
                "de R$ 1.001/mês a R$ 2.000/mês"
            )
            .when(
                F.col("faixa_salarial") == "de R$ 25.001/mês a R$ 3000/mês",
                "de R$ 25.001/mês a R$ 30.000/mês"
            )
            .otherwise(F.col("faixa_salarial"))
        )
    )

df_silver = (
    criar_silver(df_2023, mapa_2023, "2023")
    .unionByName(criar_silver(df_2024, mapa_2024, "2024"))
    .unionByName(criar_silver(df_2025, mapa_2025, "2025_2026"))
)

cloud_norm = F.lower(F.trim(F.col("cloud_preferida")))

df_silver = (
    df_silver
    .withColumn(
        "linguagem_preferida",
        F.when(
            F.col("ano_pesquisa") == "2025_2026",
            normalizar_lista("linguagem_preferida")
        ).otherwise(
            normalizar_tecnologia(F.col("linguagem_preferida"))
        )
    )
    .withColumn(
        "bancos_dados",
        normalizar_lista("bancos_dados")
    )
    .withColumn(
        "ferramenta_bi",
        normalizar_lista("ferramenta_bi")
    )
    .withColumn(
        "bi_preferida",
        normalizar_tecnologia(F.col("bi_preferida"))
    )
    .withColumn(
        "cloud_preferida",
        F.when(cloud_norm == "amazon web services (aws)", "AWS")
        .when(cloud_norm == "google cloud (gcp)", "GCP")
        .when(cloud_norm == "azure (microsoft)", "Azure")
        .when(cloud_norm.isin("databricks", "datadricks", "databriks"), "Databricks")
        .when(cloud_norm.isin("oracle", "oracle cloud"), "Oracle Cloud")
        .when(cloud_norm.isin("ibm", "ibm cloud"), "IBM Cloud")
        .when(
            cloud_norm.isin(
                "própria",
                "propria",
                "cloud própria",
                "cloud propria",
                "owncloud"
            ),
            "Cloud própria"
        )
        .when(
            cloud_norm.isin(
                "on prem",
                "on premise",
                "on-premise",
                "on-premises"
            ),
            "On-premises"
        )
        .when(
            cloud_norm.rlike(
                r"não sei|nao sei|não tenho preferência|nao tenho preferencia|"
                r"sem preferência|sem preferencia|qualquer uma|tanto faz|"
                r"não conheço|nao conheco|nao uso|não uso|^\.$"
            ),
            "Sem preferência / Não sei opinar"
        )
        .otherwise(F.trim(F.col("cloud_preferida")))
    )
)

bi_norm = F.lower(F.trim(F.col("bi_preferida")))

df_silver = df_silver.withColumn(
    "bi_preferida",
    F.when(
        bi_norm.rlike(
            r"não tenho preferência|nao tenho preferencia|"
            r"não sei opinar|nao sei opinar|^não sei$|^nao sei$|^nenhuma$"
        ),
        "Sem preferência / Não sei opinar"
    ).otherwise(F.col("bi_preferida"))
)

ordem_faixas = F.create_map(
    F.lit("Menos de R$ 1.000/mês"), F.lit(1),
    F.lit("de R$ 1.001/mês a R$ 2.000/mês"), F.lit(2),
    F.lit("de R$ 2.001/mês a R$ 3.000/mês"), F.lit(3),
    F.lit("de R$ 3.001/mês a R$ 4.000/mês"), F.lit(4),
    F.lit("de R$ 4.001/mês a R$ 6.000/mês"), F.lit(5),
    F.lit("de R$ 6.001/mês a R$ 8.000/mês"), F.lit(6),
    F.lit("de R$ 8.001/mês a R$ 12.000/mês"), F.lit(7),
    F.lit("de R$ 12.001/mês a R$ 16.000/mês"), F.lit(8),
    F.lit("de R$ 16.001/mês a R$ 20.000/mês"), F.lit(9),
    F.lit("de R$ 20.001/mês a R$ 25.000/mês"), F.lit(10),
    F.lit("de R$ 25.001/mês a R$ 30.000/mês"), F.lit(11),
    F.lit("de R$ 30.001/mês a R$ 40.000/mês"), F.lit(12),
    F.lit("Acima de R$ 40.001/mês"), F.lit(13)
)

prioridade_ia = F.col("ia_prioridade_empresa")
texto_ia = F.lower(F.col("uso_ia_trabalho"))

df_silver = (
    df_silver
    .withColumn(
        "id_unico",
        F.when(
            F.col("id_respondente").isNotNull(),
            F.concat_ws("_", F.col("ano_pesquisa"), F.col("id_respondente"))
        )
    )
    .withColumn(
        "ordem_faixa_salarial",
        F.element_at(ordem_faixas, F.col("faixa_salarial"))
    )
    .withColumn(
        "ia_prioridade_categoria",
        F.when(prioridade_ia.isNull(), F.lit(None).cast("string"))
        .when(
            prioridade_ia.startswith("Sim, é nossa principal prioridade"),
            "Principal prioridade"
        )
        .when(
            prioridade_ia.startswith("Sim, está entre nossas principais prioridades"),
            "Alta prioridade"
        )
        .when(
            prioridade_ia.startswith("Mais ou menos"),
            "Prioridade moderada"
        )
        .when(
            prioridade_ia.startswith("Não é uma iniciativa"),
            "Não é prioridade"
        )
        .when(
            prioridade_ia.startswith("Não sei opinar"),
            "Não sabe opinar"
        )
        .otherwise("Outros")
    )
    .withColumn(
        "adocao_ia_trabalho",
        F.when(F.col("uso_ia_trabalho").isNull(), F.lit(None).cast("string"))
        .when(
            texto_ia.contains("utilizo apenas soluções gratuitas")
            | texto_ia.contains("utilizo soluções pagas de ai generativa")
            | texto_ia.contains("utilizo soluções no estilo")
            | texto_ia.contains("utilizo soluções de ai para código"),
            "Utiliza IA"
        )
        .when(
            texto_ia.contains("não utilizo nenhum tipo de solução"),
            "Não utiliza IA"
        )
        .otherwise("Outros")
    )
)

In [9]:
auditoria_transformacoes = (
    df_silver
    .agg(
        F.sum(
            F.when(
                F.coalesce(F.col("faixa_salarial_raw"), F.lit("<NULL>"))
                != F.coalesce(F.col("faixa_salarial"), F.lit("<NULL>")),
                1
            ).otherwise(0)
        ).alias("faixas_salariais_corrigidas"),
        F.sum(
            F.when(
                F.coalesce(F.col("bancos_dados_raw"), F.lit("<NULL>"))
                != F.coalesce(F.col("bancos_dados"), F.lit("<NULL>")),
                1
            ).otherwise(0)
        ).alias("bancos_dados_alterados"),
        F.sum(
            F.when(
                F.coalesce(F.col("cloud_preferida_raw"), F.lit("<NULL>"))
                != F.coalesce(F.col("cloud_preferida"), F.lit("<NULL>")),
                1
            ).otherwise(0)
        ).alias("clouds_padronizadas"),
        F.sum(
            F.when(
                F.coalesce(F.col("linguagem_preferida_raw"), F.lit("<NULL>"))
                != F.coalesce(F.col("linguagem_preferida"), F.lit("<NULL>")),
                1
            ).otherwise(0)
        ).alias("linguagens_padronizadas"),
        F.sum(
            F.when(
                F.coalesce(F.col("ferramenta_bi_raw"), F.lit("<NULL>"))
                != F.coalesce(F.col("ferramenta_bi"), F.lit("<NULL>")),
                1
            ).otherwise(0)
        ).alias("ferramentas_bi_padronizadas")
    )
)

auditoria_transformacoes.show(truncate=False)

+---------------------------+----------------------+-------------------+-----------------------+---------------------------+
|faixas_salariais_corrigidas|bancos_dados_alterados|clouds_padronizadas|linguagens_padronizadas|ferramentas_bi_padronizadas|
+---------------------------+----------------------+-------------------+-----------------------+---------------------------+
|2                          |7513                  |9416               |1431                   |10216                      |
+---------------------------+----------------------+-------------------+-----------------------+---------------------------+


In [10]:
(
    df_silver
    .filter(
        F.coalesce(F.col("faixa_salarial_raw"), F.lit("<NULL>"))
        != F.coalesce(F.col("faixa_salarial"), F.lit("<NULL>"))
    )
    .select(
        "ano_pesquisa",
        "faixa_salarial_raw",
        "faixa_salarial"
    )
    .distinct()
    .show(20, truncate=False)
)

(
    df_silver
    .filter(
        F.coalesce(F.col("cloud_preferida_raw"), F.lit("<NULL>"))
        != F.coalesce(F.col("cloud_preferida"), F.lit("<NULL>"))
    )
    .groupBy(
        "ano_pesquisa",
        "cloud_preferida_raw",
        "cloud_preferida"
    )
    .count()
    .orderBy(
        "ano_pesquisa",
        F.desc("count")
    )
    .show(30, truncate=False)
)

+------------+------------------------------+--------------------------------+
|ano_pesquisa|faixa_salarial_raw            |faixa_salarial                  |
+------------+------------------------------+--------------------------------+
|2023        |de R$ 101/mês a R$ 2.000/mês  |de R$ 1.001/mês a R$ 2.000/mês  |
|2025_2026   |de R$ 25.001/mês a R$ 3000/mês|de R$ 25.001/mês a R$ 30.000/mês|
+------------+------------------------------+--------------------------------+

+------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------+-----+
|ano_pesquisa|cloud_preferida_raw                                                                                                                                                                                                       

Categorias após padronização

In [11]:
for campo in [
    "genero",
    "senioridade",
    "cloud_preferida",
    "ia_prioridade_categoria",
    "adocao_ia_trabalho"
]:
    print(campo)

    (
        df_silver
        .groupBy("ano_pesquisa", campo)
        .count()
        .orderBy(
            "ano_pesquisa",
            F.desc("count")
        )
        .show(25, truncate=False)
    )

genero
+------------+--------------------+-----+
|ano_pesquisa|genero              |count|
+------------+--------------------+-----+
|2023        |Masculino           |3975 |
|2023        |Feminino            |1293 |
|2023        |Prefiro não informar|16   |
|2023        |Outro               |9    |
|2024        |Masculino           |3967 |
|2024        |Feminino            |1225 |
|2024        |Prefiro não informar|15   |
|2024        |Outro               |8    |
|2025_2026   |Masculino           |2707 |
|2025_2026   |Feminino            |767  |
|2025_2026   |Prefiro não informar|13   |
|2025_2026   |Outro               |7    |
+------------+--------------------+-----+

senioridade
+------------+-------------------+-----+
|ano_pesquisa|senioridade        |count|
+------------+-------------------+-----+
|2023        |NULL               |1436 |
|2023        |Sênior             |1419 |
|2023        |Pleno              |1392 |
|2023        |Júnior             |1046 |
|2024        |Sênior 

## geral antes de gravar na Silver

In [12]:
colunas_obrigatorias = [
    "id_respondente",
    "id_unico",
    "idade",
    "genero",
    "regiao",
    "cargo_atual",
    "senioridade",
    "faixa_salarial",
    "ordem_faixa_salarial",
    "linguagem_preferida",
    "bancos_dados",
    "cloud_preferida",
    "ferramenta_bi",
    "bi_preferida",
    "ia_prioridade_categoria",
    "adocao_ia_trabalho",
    "ano_pesquisa"
]

faltantes = [
    coluna
    for coluna in colunas_obrigatorias
    if coluna not in df_silver.columns
]

if faltantes:
    raise ValueError(f"Colunas obrigatórias ausentes: {faltantes}")

validacao_estrutura = (
    df_silver
    .agg(
        F.count("*").alias("total_registros"),
        F.countDistinct("id_unico").alias("ids_unicos"),
        F.sum(
            F.when(F.col("id_unico").isNull(), 1).otherwise(0)
        ).alias("ids_nulos"),
        F.sum(
            F.when(
                (F.col("idade") < 15) | (F.col("idade") > 100),
                1
            ).otherwise(0)
        ).alias("idades_suspeitas")
    )
)

validacao_estrutura.show(truncate=False)

+---------------+----------+---------+----------------+
|total_registros|ids_unicos|ids_nulos|idades_suspeitas|
+---------------+----------+---------+----------------+
|14002          |14002     |0        |0               |
+---------------+----------+---------+----------------+


## Gravando na Silver

In [13]:
spark.sql(f"DROP TABLE IF EXISTS {SILVER_TABLE}")

(
    df_silver
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .option("path", SILVER_PATH)
    .partitionBy("ano_pesquisa")
    .saveAsTable(SILVER_TABLE)
)

In [14]:
df_check = spark.table(SILVER_TABLE)

(
    df_check
    .groupBy("ano_pesquisa")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("id_respondente").alias("ids_unicos"),
        F.sum(
            F.when(F.col("id_respondente").isNull(), 1).otherwise(0)
        ).alias("ids_nulos"),
        F.sum(
            F.when(
                (F.col("idade") < 15) | (F.col("idade") > 100),
                1
            ).otherwise(0)
        ).alias("idades_suspeitas")
    )
    .orderBy("ano_pesquisa")
    .show()
)

df_check.select(
    "ano_pesquisa",
    "genero",
    "cargo_atual",
    "senioridade",
    "faixa_salarial",
    "ordem_faixa_salarial",
    "cloud_preferida",
    "ia_prioridade_categoria",
    "adocao_ia_trabalho"
).show(10, truncate=False)

+------------+---------+----------+---------+----------------+
|ano_pesquisa|registros|ids_unicos|ids_nulos|idades_suspeitas|
+------------+---------+----------+---------+----------------+
|        2023|     5293|      5293|        0|               0|
|        2024|     5215|      5215|        0|               0|
|   2025_2026|     3494|      3494|        0|               0|
+------------+---------+----------+---------+----------------+

+------------+---------+-------------------------------------+-------------------+--------------------------------+--------------------+---------------+-----------------------+------------------+
|ano_pesquisa|genero   |cargo_atual                          |senioridade        |faixa_salarial                  |ordem_faixa_salarial|cloud_preferida|ia_prioridade_categoria|adocao_ia_trabalho|
+------------+---------+-------------------------------------+-------------------+--------------------------------+--------------------+---------------+--------------

In [ ]:
%stop_session